# ColdSite-DTI — KIBA half of the 24-run grid (Kaggle, T4 x2)

Trains the **12 KIBA cells** (4 splits x 3 seeds), regression, ColdSite-DTI.
The 12 DAVIS cells run separately on Colab; together they are STATUS.md item 6.

## Before you run

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (Settings -> Accelerator) |
| Internet | **On** (Settings -> Internet) — needed for the clone and the DeepDTA files |
| How to run | **Save & Run All (Commit)** — batch mode, up to 12 h unattended |

Do **not** use an interactive session for the long run: Kaggle idles it out after
about 20 minutes and a half-finished cell is lost.

## Why two processes

The model is single-GPU, so a second T4 does nothing on its own. This notebook runs
**two `run_grid` processes in parallel**, one pinned to each GPU, splitting the four
split types 2/2. No code changes — `run_grid` already takes `--splits`.

| | GPU 0 | GPU 1 |
|---|---|---|
| splits | `random`, `cold_drug` | `cold_target`, `cold_pair` |
| cells | 6 | 6 |

## What to expect

KIBA's training split is **82,778 rows against DAVIS's 21,039 — 3.9x**. At the ~2.3
min/epoch measured for DAVIS on a T4, a KIBA epoch is ~9 min and a cell that early-stops
around epoch 33 takes **~5 hours**. So one 12-hour commit finishes roughly **2 cells per
GPU, 4 total**. Expect ~3 commits to finish all 12.

**Resumable.** Finished cells are skipped on the next run, and a cell interrupted by the
12-hour limit is *retrained* rather than banked as done (commit `d9a03c1`). There is no
within-cell resume, so a cell cut off at hour 11 is lost — which is why the cells-per-run
arithmetic above matters.


## 1. Check the GPUs

Stops unless CUDA is present. Reports how many devices, because the parallel launch
below falls back to one process if there is only one.


In [ ]:
import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU T4 x2.'

N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)
print('devices:', N_GPU)

# batch 64 needs ~8.7 GB on a 1000-residue protein; a T4 has 15.6 GB.
BATCH = 64 if torch.cuda.get_device_properties(0).total_memory/1e9 >= 14 else 16
print('batch  :', BATCH)
if N_GPU < 2:
    print('\nOnly one GPU — the notebook will run a single process over all 12 cells.')


## 2. Clone the repo

**Pinned to `main` on the fork.** The DAVIS cells on Colab are trained from the same
branch; if the two halves came from different code they would not be one grid.


In [ ]:
import os, subprocess

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC  = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt

# Results live outside the clone so a re-clone never destroys finished cells.
RESULTS = f'{WORK}/results'
os.makedirs(RESULTS, exist_ok=True)
print()
!git log --oneline -1
print('results ->', RESULTS)
print('existing checkpoints:',
      len([f for f in os.listdir(RESULTS) if f.endswith('.pt')]), '/ 12')


## 3. Fetch the DeepDTA source files

Gitignored — they belong to DeepDTA. KIBA and DAVIS are both fetched because
`build_splits` builds both datasets in one pass; only KIBA is trained here.


In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'

for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}

!python -m src.data.load_data


Expected, exactly:

```
davis: 30056 measured pairs, 68 unique drugs, 442 unique targets, Y range [5.000, 10.796]
kiba: 118254 measured pairs, 2111 unique drugs, 229 unique targets, Y range [0.000, 17.200]
```


## 4. Build the splits — and verify they match Colab's

This is the check that matters most in this notebook. The KIBA cells trained here have
to come from the *same* splits as the DAVIS cells trained on Colab, or the grid is two
different experiments wearing one name. `build_splits` is deterministic from a fixed
seed, so the counts below must match exactly — the cell asserts it rather than trusting it.


In [ ]:
!python -m src.data.build_splits

import pandas as pd

# Verified identical on a MacBook and on Colab. Any difference here means the
# split builder is not reproducing, and nothing trained after it is comparable.
EXPECTED = {
    'random':      (82778, 11825, 23651),
    'cold_drug':   (83807, 12073, 22374),
    'cold_target': (85452, 10701, 22101),
    'cold_pair':   (58041,  1334,  4375),
}

problems = []
for split, expected in EXPECTED.items():
    got = tuple(len(pd.read_csv(f'data/splits/kiba/{split}/{part}.csv'))
                for part in ('train', 'valid', 'test'))
    flag = 'OK' if got == expected else 'MISMATCH'
    if got != expected:
        problems.append(f'{split}: expected {expected}, got {got}')
    print(f'{split:12s} {str(got):28s} {flag}')

assert not problems, (
    'KIBA splits do not match the ones the DAVIS half was built from:\n  '
    + '\n  '.join(problems)
    + '\nDo not train on these — the two halves would not be one grid.')
print('\nSplits match the DAVIS half. Safe to train.')


## 5. Preflight


In [ ]:
!python -m src.model.run_grid --preflight --datasets kiba --results-dir {RESULTS} 2>&1 | tail -5


## 5b. Profile one forward pass (about a minute)

Runs while both GPUs are still idle, so the numbers are not contaminated by training.

The model discards the protein encoder's self-attention weights — `coldsite_dti.py`
line 54 assigns them to `_protein_self_attn` and nothing reads it — but the encoder
still asks for them, which materialises a `(batch, len, len)` tensor and forces
`nn.MultiheadAttention` off its fused kernel. The gap between the two `attention`
rows below is what that costs.

If the `BiLSTM branch` row dominates both instead, the ~660-1000 sequential timesteps
are the real cost and no attention change is worth making.


In [ ]:
!python -u -m src.model.profile_forward --batch-size {BATCH} --protein-len 1000


## 5c. Restore results from a previous run (optional)

`/kaggle/working` starts empty in a fresh container, so finished cells do not carry over
between commits or accounts on their own. This cell copies them back in, and `run_grid`
then skips whatever is already complete.

**To use it:** open the earlier run's saved version, download its `results/` folder, upload
it as a Kaggle **Dataset**, then attach that dataset here (right panel -> Add Input) and put
its path in `RESTORE_FROM`. Leave it `None` on a first run.

Only `.pt` and `*_results.json` files are copied, and only when the destination does not
already have them -- so re-running this cell never overwrites work done in this session.
A checkpoint that arrives without its results JSON is an interrupted cell and gets
retrained, which is the correct outcome rather than a half-trained model treated as done.


In [ ]:
import shutil, glob

# e.g. '/kaggle/input/coldsite-kiba-results'   (None on a first run)
RESTORE_FROM = None

if RESTORE_FROM:
    assert os.path.isdir(RESTORE_FROM), f'not a directory: {RESTORE_FROM}'
    copied = skipped = 0
    for src in glob.glob(f'{RESTORE_FROM}/**/*', recursive=True):
        name = os.path.basename(src)
        if not (name.endswith('.pt') or name.endswith('_results.json')
                or name.endswith('_history.json')):
            continue
        dst = os.path.join(RESULTS, name)
        if os.path.exists(dst):
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    print(f'restored {copied} file(s), left {skipped} already present')

    # What run_grid will actually make of them.
    ckpt = {os.path.basename(p)[:-3].replace('coldsite_dti_', '')
            for p in glob.glob(f'{RESULTS}/*.pt')}
    done = {os.path.basename(p).replace('_results.json', '')
            for p in glob.glob(f'{RESULTS}/*_results.json')}
    print(f'complete cells    : {len(ckpt & done)}  (will be skipped)')
    print(f'interrupted cells : {len(ckpt - done)}  (will be retrained)')
else:
    print('RESTORE_FROM is None — starting from whatever is already in', RESULTS)


## 6. Train — two processes, one per GPU

Each process validates its own first unfinished cell end to end before launching the
rest. Output streams live, tagged by GPU, and is also written to `gpu0.log` / `gpu1.log`
so it survives if the cell output is cleared.


In [ ]:
import subprocess, threading, os

ASSIGNMENT = ([('0', 'random,cold_drug'), ('1', 'cold_target,cold_pair')]
              if N_GPU >= 2 else
              [('0', 'random,cold_drug,cold_target,cold_pair')])


def stream(gpu, proc, path):
    """Print every line as it arrives, tagged, and tee it to a log file.

    A reader thread per process, rather than polling `tail` on a timer: the
    processes run in parallel so neither can own the cell's output, but both
    can write to it line by line as Colab does with a single foreground run.
    """
    with open(path, 'w') as log:
        for line in proc.stdout:
            print(f'[GPU{gpu}] {line}', end='', flush=True)
            log.write(line)
            log.flush()


procs = []
for gpu, splits in ASSIGNMENT:
    # PYTHONUNBUFFERED so lines arrive as they are printed rather than in
    # 4-8 KB blocks -- without it a healthy run looks hung for hours.
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': gpu, 'PYTHONUNBUFFERED': '1'}
    cmd = ['python', '-u', '-m', 'src.model.run_grid',
           '--datasets', 'kiba',
           '--splits', splits,
           '--batch-size', str(BATCH),
           '--results-dir', RESULTS]
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    thread = threading.Thread(target=stream,
                              args=(gpu, proc, f'{WORK}/gpu{gpu}.log'),
                              daemon=True)
    thread.start()
    procs.append((gpu, proc, thread))
    print(f'GPU {gpu} -> {splits}', flush=True)

print(flush=True)
for gpu, proc, thread in procs:
    thread.join()
    print(f'GPU {gpu} exited with code {proc.wait()}', flush=True)


## 7. What landed

A checkpoint **without** a matching results JSON is an interrupted cell, not a finished
one — `run_grid` will retrain it on the next commit.


In [ ]:
import glob

ckpt = glob.glob(f'{RESULTS}/*kiba*.pt')
res  = glob.glob(f'{RESULTS}/*kiba*_results.json')
print('checkpoints :', len(ckpt), '/ 12')
print('run results :', len(res), '/ 12')

# Checkpoints carry a 'coldsite_dti_' prefix that results files do not;
    # comparing the raw stems marked every finished cell as interrupted.
    stems = {os.path.basename(p)[:-3].replace('coldsite_dti_', '') for p in ckpt}
done  = {os.path.basename(p).replace('_results.json', '') for p in res}
print('interrupted :', len(stems - done))

import json as _json
for path in sorted(res):
    payload = _json.load(open(path))
    m = payload.get('test_metrics', {})
    print(f"  {payload.get('split'):12s} seed{payload.get('seed')}  "
          f"ci={m.get('ci', float('nan')):.4f}  mse={m.get('mse', float('nan')):.4f}")

print()
!python -m src.model.run_grid --preflight --datasets kiba --results-dir {RESULTS} 2>&1 | tail -3


## 8. Getting the results back

Everything under `/kaggle/working` is saved as this notebook version's output. Download
`results/` from the version's Output tab and drop the `.pt` and `*_results.json` files
into the Colab Drive folder `coldsite-grid24-results`, alongside the DAVIS cells.

The downstream steps — `run_faithfulness`, `run_ladder`, `run_audit` — read all 24 cells
from one directory, so they only work once both halves sit together.
